# Notebook 08 — Mini-Project: Michaelis-Menten Kinetics Simulator

**Module:** 07 — Biochemistry and Molecular Biology  
**Notebook:** 08 of 10  
**Portfolio artifact:** `portfolio/module07_michaelis_menten_kinetics.png`  
**Prerequisites:** NB03 (enzyme kinetics) — implement from memory  
**Time estimate:** 90 minutes

> **Implementation rule:** All functions in this notebook must be written from memory.
> Close NB03 before starting. This is the Pass-3 reconstruction for Module 07's
> core algorithm.

---
## Project Brief

A single 4-panel figure demonstrating mastery of enzyme kinetics. A stranger who
knows biochemistry but has not seen this code should be able to understand all four
panels from their titles, axis labels, and annotations alone.

| Panel | Content | Key concept |
|-------|---------|-------------|
| A | MM hyperbola: v vs. [S], annotated with Km and Vmax | Basic kinetics |
| B | Lineweaver-Burk: 1/v vs. 1/[S], linear fit with annotated intercepts | Linearisation |
| C | Three inhibition models on one axes | Inhibition classification |
| D | Hill equation: v vs. [S] for n=0.5, 1, 2, 4 | Cooperativity |

---
## Implementation — From Memory

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

os.makedirs('../../portfolio', exist_ok=True)

In [ ]:
# ── IMPLEMENT FROM MEMORY ─────────────────────────────────────────────────────
# Implement all functions before running the figure cell.

def michaelis_menten(S, Vmax, Km):
    """Standard MM equation: v = Vmax*S/(Km+S)"""
    pass  # TODO

def competitive_inhibition(S, Vmax, Km, I, Ki):
    """Competitive: apparent Km increases, Vmax unchanged."""
    pass  # TODO

def noncompetitive_inhibition(S, Vmax, Km, I, Ki):
    """Non-competitive: apparent Vmax decreases, Km unchanged."""
    pass  # TODO

def uncompetitive_inhibition(S, Vmax, Km, I, Ki):
    """Uncompetitive: both Km and Vmax decrease by same factor."""
    pass  # TODO

def hill_equation(S, Vmax, Kd, n):
    """Hill equation: v = Vmax * S^n / (Kd^n + S^n)"""
    pass  # TODO

def lineweaver_burk_params(S_vals, v_vals):
    """
    Fit a Lineweaver-Burk double-reciprocal plot.
    Returns (Km, Vmax) from the linear fit to 1/v vs. 1/[S].
    """
    pass  # TODO

# Parameters
Vmax, Km = 10.0, 2.0
S_range = np.linspace(0.01, 20, 300)
I, Ki = 5.0, 3.0

# Sanity checks — these should pass before you continue
print("Sanity checks:")
v_half = michaelis_menten(Km, Vmax, Km)
print(f"  v at [S]=Km: {v_half}  (expected: {Vmax/2})")
v_sat = michaelis_menten(100*Km, Vmax, Km)
print(f"  v at [S]=100*Km: {v_sat:.3f}  (expected ≈ {Vmax})")
v_hill1 = hill_equation(Km, Vmax, Km, 1.0)
print(f"  Hill n=1 at [S]=Kd: {v_hill1}  (should equal MM: {Vmax/2})")

In [ ]:
# ── Simulate experimental data for Lineweaver-Burk ──────────────────────────
rng = np.random.default_rng(42)
S_exp = np.array([0.2, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0])
v_true = michaelis_menten(S_exp, Vmax, Km)
v_noisy = np.clip(v_true + rng.normal(0, 0.25, len(S_exp)), 0.05, None)
Km_lb, Vmax_lb = lineweaver_burk_params(S_exp, v_noisy)
print(f"Lineweaver-Burk recovered: Km={Km_lb:.2f} (true={Km}), Vmax={Vmax_lb:.2f} (true={Vmax})")

In [ ]:
# ── Compute all curves ───────────────────────────────────────────────────────
v_no_inhib   = michaelis_menten(S_range, Vmax, Km)
v_comp       = competitive_inhibition(S_range, Vmax, Km, I, Ki)
v_noncomp    = noncompetitive_inhibition(S_range, Vmax, Km, I, Ki)
v_uncomp     = uncompetitive_inhibition(S_range, Vmax, Km, I, Ki)

hill_curves  = {n: hill_equation(S_range, Vmax, Km, n) for n in [0.5, 1.0, 2.0, 4.0]}

In [ ]:
# ── Portfolio figure ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.33)
axes = [fig.add_subplot(gs[i//2, i%2]) for i in range(4)]

# ── Panel A: MM hyperbola ─────────────────────────────────────────────────
ax = axes[0]
ax.plot(S_range, v_no_inhib, color='steelblue', lw=2.5)
ax.scatter(S_exp, v_noisy, color='tomato', s=40, zorder=5, label='Simulated data')

# Annotate Km and Vmax
ax.axhline(Vmax, color='gray', ls=':', lw=1)
ax.axhline(Vmax/2, color='gray', ls=':', lw=1)
ax.axvline(Km, color='gray', ls=':', lw=1)
ax.annotate('', xy=(Km, Vmax/2), xytext=(0, Vmax/2),
            arrowprops=dict(arrowstyle='<->', color='seagreen', lw=1.5))
ax.text(Km/2, Vmax/2 + 0.3, f'Km = {Km} mM', ha='center', color='seagreen', fontsize=8)
ax.text(18, Vmax * 1.04, f'Vmax = {Vmax} µmol/min', ha='right', color='gray', fontsize=8)
ax.set_xlabel('[S] (mM)'); ax.set_ylabel('v (µmol/min)')
ax.set_title('A. Michaelis-Menten kinetics\nv = Vmax·[S]/(Km + [S])')
ax.legend(fontsize=8)

# ── Panel B: Lineweaver-Burk ──────────────────────────────────────────────
ax = axes[1]
inv_s_exp = 1 / S_exp
inv_v_noisy = 1 / v_noisy

slope_lb = Km_lb / Vmax_lb
intercept_lb = 1 / Vmax_lb

x_lb = np.linspace(-0.6, 6, 200)
y_lb = slope_lb * x_lb + intercept_lb

ax.plot(x_lb, y_lb, color='steelblue', lw=2)
ax.scatter(inv_s_exp, inv_v_noisy, color='tomato', s=40, zorder=5)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)

ax.scatter([-1/Km_lb], [0], color='seagreen', s=60, zorder=6)
ax.scatter([0], [1/Vmax_lb], color='orange', s=60, zorder=6)
ax.annotate(f'x-intercept = −1/Km = {-1/Km_lb:.2f}',
            xy=(-1/Km_lb, 0), xytext=(0.5, -0.05),
            arrowprops=dict(arrowstyle='->', color='seagreen'), fontsize=7, color='seagreen')
ax.annotate(f'y-intercept = 1/Vmax = {1/Vmax_lb:.3f}',
            xy=(0, 1/Vmax_lb), xytext=(1.5, 0.25),
            arrowprops=dict(arrowstyle='->', color='orange'), fontsize=7, color='orange')
ax.set_xlabel('1/[S] (mM⁻¹)'); ax.set_ylabel('1/v (min/µmol)')
ax.set_title('B. Lineweaver-Burk (double-reciprocal)\nslope = Km/Vmax')
ax.set_xlim(-0.6, 6); ax.set_ylim(-0.08, 0.8)

# ── Panel C: Inhibition models ────────────────────────────────────────────
ax = axes[2]
ax.plot(S_range, v_no_inhib, color='steelblue', lw=2, label=f'No inhibitor')
ax.plot(S_range, v_comp, color='tomato', lw=2, ls='--',
        label=f'Competitive (Km↑, Vmax unchanged)')
ax.plot(S_range, v_noncomp, color='seagreen', lw=2, ls=':',
        label=f'Non-competitive (Km unchanged, Vmax↓)')
ax.plot(S_range, v_uncomp, color='orange', lw=2, ls='-.',
        label=f'Uncompetitive (both↓, parallel LB lines)')
ax.axhline(Vmax, color='steelblue', ls=':', lw=0.8, alpha=0.5)
ax.set_xlabel('[S] (mM)'); ax.set_ylabel('v (µmol/min)')
ax.set_title(f'C. Inhibition types ([I]={I} mM, Ki={Ki} mM)')
ax.legend(fontsize=7.5)

# ── Panel D: Hill cooperativity ───────────────────────────────────────────
ax = axes[3]
hill_colors = {0.5: 'lightblue', 1.0: 'gray', 2.0: 'orange', 4.0: 'tomato'}
for n, color in hill_colors.items():
    ls = '--' if n == 1.0 else '-'
    ax.plot(S_range, hill_curves[n], color=color, lw=2, ls=ls, label=f'n = {n}')
ax.axhline(Vmax/2, color='gray', ls=':', lw=0.8, alpha=0.6)
ax.axvline(Km, color='gray', ls=':', lw=0.8, alpha=0.6)
ax.text(Km + 0.3, Vmax/2 + 0.3, f'Kd = {Km}', fontsize=8, color='gray')
ax.set_xlabel('[S] (mM)'); ax.set_ylabel('v (µmol/min)')
ax.set_title('D. Hill equation: cooperativity\nn>1 = sigmoidal (positive cooperativity)')
ax.legend(fontsize=8)

# Panel labels
for ax, label in zip(axes, ['A', 'B', 'C', 'D']):
    ax.text(-0.10, 1.05, label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top')

fig.suptitle('Enzyme Kinetics: Michaelis-Menten and Extensions\n'
             'Module 07 — Biochemistry and Molecular Biology Portfolio Figure',
             fontsize=12, y=1.01)

plt.savefig('../../portfolio/module07_michaelis_menten_kinetics.png',
            dpi=150, bbox_inches='tight')
print("Figure saved to portfolio/module07_michaelis_menten_kinetics.png")
plt.show()

---
## Pass-3 Record

**Date completed:** ____________________  
**Implemented from memory (no copying from NB03)?** [ ] Yes  [ ] Needed to check ___________

---
## Reflection

> *[Write 3–5 sentences: explain MM kinetics to a software engineer who has never
> taken biochemistry. Which panel of this figure would you use as a teaching tool
> for that conversation, and why?]*

---
*Next: `09_paper_reading_session.ipynb`*